# 05 — Baselines
Three baselines, each must be beaten by what follows: complexity has to pay for itself.
1. **Univariate z-score on s11** — no fitting at all; sets the floor.
2. **Mahalanobis on raw sensors (no PCA)** — isolates what the latent space adds.
3. **Supervised skyline** — gradient boosting on identical features WITH labels; the ceiling.
   Run on FD004 no-buffer only (everything saturates elsewhere).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
from src.loaders import prepare, SENSORS, HEALTHY_RUL
from src import metrics

res = {}

In [2]:
# Baseline 1+2 on FD001 and FD004 (normalized splits from notebook 04 pipeline)
for fd in ['FD001', 'FD004']:
    d = prepare(fd, seed=42)
    fit, val = d['fit'], d['val']
    # B1: |z| of s11 (data already z-scored on healthy stats -> just abs)
    res[(fd, 'B1 z-s11')] = {p: metrics.auroc(val.s11.abs(), val.rul, use_buffer=b)
                             for p, b in [('buffer', True), ('no-buffer', False)]}
    # B2: Mahalanobis on the 14 raw (normalized) sensors, healthy Gaussian
    H = fit[fit.rul > HEALTHY_RUL][SENSORS]
    VI = np.linalg.inv(np.cov(H.T.values) + 1e-6 * np.eye(len(SENSORS)))
    dlt = val[SENSORS].values - H.mean().values
    md = np.sqrt(np.einsum('ij,jk,ik->i', dlt, VI, dlt))
    res[(fd, 'B2 Mahal-sensors')] = {p: metrics.auroc(md, val.rul, use_buffer=b)
                                     for p, b in [('buffer', True), ('no-buffer', False)]}

In [3]:
# Baseline 3: supervised skyline, FD004 no-buffer (labels y = RUL<30 USED in training)
from sklearn.ensemble import HistGradientBoostingClassifier
d = prepare('FD004', seed=42)
Xf, yf = d['fit'][SENSORS], (d['fit'].rul < 30).astype(int)
clf = HistGradientBoostingClassifier(random_state=0).fit(Xf, yf)
p = clf.predict_proba(d['val'][SENSORS])[:, 1]
res[('FD004', 'B3 skyline (supervised)')] = {
    'buffer': metrics.auroc(p, d['val'].rul, use_buffer=True),
    'no-buffer': metrics.auroc(p, d['val'].rul, use_buffer=False)}

In [4]:
tab = pd.DataFrame(res).T.round(3)
tab.to_csv('../results/tables/05_baselines.csv')
tab

buffer  no-buffer
FD001 B1 z-s11                  0.991      0.972
      B2 Mahal-sensors          0.971      0.928
FD004 B1 z-s11                  0.991      0.969
      B2 Mahal-sensors          0.994      0.970
      B3 skyline (supervised)   1.000      0.991

Readings: (i) the no-model baseline is already ≈0.99 on FD001-buffer — **the buffer protocol
saturates**, which is why FD004 no-buffer is the pre-registered primary setting; (ii) the
skyline bounds what any unsupervised method can hope for; all later results are quoted
as a fraction of it.